In [ ]:
# [Cell 1] 모듈 로드
%load_ext autoreload
%autoreload 2

import torch
import ipywidgets as widgets
from IPython.display import display
from modules.hardware import AGVHardware
from modules.driving_logic import LineTrackingBrain
from modules.mission_manager import MissionManager

# 1. 모델 로드 (경로는 본인 환경에 맞게 수정)
model = torch.load('best_steering_model_xy.pth')
device = torch.device('cuda')
model = model.to(device)
model.eval().half()

# 2. 객체 조립
agv = AGVHardware()
brain = LineTrackingBrain(model, device)
manager = MissionManager(agv, brain)

print("시스템 준비 완료!")

In [ ]:
# [Cell 2] UI (슬라이더) 생성 및 Context 연결
# RobotMoving.py에 있던 그 슬라이더들입니다.

speed_slider = widgets.FloatSlider(value=0.15, min=0.0, max=0.5, step=0.01, description='Speed')
steering_gain = widgets.FloatSlider(value=0.04, min=0.0, max=0.2, step=0.001, description='St. Gain')
steering_dgain = widgets.FloatSlider(value=0.0, min=0.0, max=0.5, step=0.001, description='St. DGain')
steering_bias = widgets.FloatSlider(value=0.0, min=-0.3, max=0.3, step=0.01, description='Bias')

image_widget = widgets.Image(format='jpeg', width=224, height=224)
x_slider = widgets.FloatSlider(min=-1.0, max=1.0, description='x')
y_slider = widgets.FloatSlider(min=0, max=1.0, orientation='vertical', description='y')

# 제어 버튼
start_btn = widgets.Button(description='Start Tracking', button_style='success')
stop_btn = widgets.Button(description='Stop', button_style='danger')

def on_start(_): manager.set_state("TRACKING")
def on_stop(_): manager.set_state("IDLE")

start_btn.on_click(on_start)
stop_btn.on_click(on_stop)

display(widgets.HBox([image_widget, y_slider]), x_slider)
display(speed_slider, steering_gain, steering_dgain, steering_bias)
display(widgets.HBox([start_btn, stop_btn]))

In [ ]:
# [Cell 3] 메인 루프 실행
import time

try:
    while True:
        # 1. 슬라이더 값을 Context에 실시간 반영 (동기화)
        ctx = manager.context
        ctx.speed_gain = speed_slider.value
        ctx.steering_gain = steering_gain.value
        ctx.steering_dgain = steering_dgain.value
        ctx.steering_bias = steering_bias.value
        
        # 2. FSM 업데이트 (여기서 주행 로직이 돕니다)
        manager.update()
        
        # 3. 화면 업데이트 (디버깅용)
        if ctx.processed_image:
            image_widget.value = ctx.processed_image
            x_slider.value = ctx.current_x
            y_slider.value = ctx.current_y
            
        time.sleep(0.01) # CPU 과부하 방지
        
except KeyboardInterrupt:
    agv.stop()
    print("종료되었습니다.")